In [1]:
import json
import jsonlines
import pandas as pd
from pathlib import Path
from openai import OpenAI
from tqdm.notebook import tqdm
from pydantic import BaseModel, Field

pd.set_option('display.max_colwidth', None)

MODEL_NAME = 'gpt-5.4-mini'
REASONING_EFFORT = 'high'
SERVICE_TIER = 'flex'

DATA_PATH_COMMENTS = '../data/results/binary/question_detection_gpt-5-4-nano_medium_flex.csv'
JSON_PATH = f'../data/results/extracted_question/relevant_question_extraction_{MODEL_NAME.replace(".", "-")}_{REASONING_EFFORT}_{SERVICE_TIER}.jsonl'
OUTPUT_PATH = f'../data/results/extracted_question/relevant_question_extraction_{MODEL_NAME.replace(".", "-")}_{REASONING_EFFORT}_{SERVICE_TIER}.csv'
USAGE_PATH = f'../data/results/usage/extracted_question/relevant_question_extraction_{MODEL_NAME.replace(".", "-")}_{REASONING_EFFORT}_{SERVICE_TIER}_usage.csv'
SAMPLED_PATH = f'../data/cleaned/sampled_data_two_steps.csv'

# Registra os tokens usados em cada chamada da API.
LOG_USAGE = True

client = OpenAI()

In [2]:
class QuestionExtraction(BaseModel):
    perguntas: list[str] = Field(default_factory=list)

In [3]:
df = pd.read_csv(DATA_PATH_COMMENTS)
df = df[df['tem_pergunta'].astype(str).str.lower().eq('true')].copy()
comments_ids = df['comment_id'].unique().tolist()
df['comment_id'].nunique(), df.shape, len(comments_ids)

(35479, (35479, 5), 35479)

In [4]:
EXTRACTION_PROMPT = """
Extraia e reformule as perguntas relevantes presentes no campo "comentario".

Você receberá:
- "titulo": título do vídeo em que o comentário foi publicado;
- "comentario_pai": comentário ao qual o comentário atual está respondendo, podendo estar vazio;
- "comentario": comentário que deve ser analisado.

Considere perguntas explícitas e dúvidas claramente implícitas. Use "titulo" e "comentario_pai" apenas como contexto para interpretar o comentário atual.

Extraia somente perguntas relacionadas à gravidez e ao contexto reprodutivo ou materno-infantil. Isso inclui, sem se limitar a, fertilidade, tentativa de engravidar, suspeita ou confirmação de gravidez, desenvolvimento gestacional, aborto, parto, pós-parto, amamentação e saúde do bebê.

Ignore perguntas claramente alheias a esse contexto, como perguntas sobre o canal, o vídeo, a apresentadora ou pessoas mencionadas no conteúdo.

Reformule cada pergunta para que seja:
- compreensível sem o comentário original;
- escrita em português claro e correto;
- fiel ao sentido original;
- composta apenas por informações presentes nos campos recebidos.

Não responda às perguntas e não acrescente informações.

Se houver mais de uma pergunta relevante, retorne cada uma separadamente. Se nenhuma pergunta relevante for encontrada, retorne uma lista vazia.

Retorne apenas um objeto JSON válido, sem explicações ou texto adicional:

{"perguntas": ["pergunta reformulada 1", "pergunta reformulada 2"]}

Caso não exista uma pergunta relevante:

{"perguntas": []}
"""

In [5]:
EXTRACTION_EXAMPLES = [
    {
        "titulo": "Exercícios para se preparar para o parto normal",
        "comentario_pai": "",
        "comentario": "Quero saber ser posso fazer com 5meses?",
        "perguntas": [
            "Gestantes com cinco meses de gravidez podem realizar exercícios de preparação para o parto normal?"
        ],
    },
    {
        "titulo": "A MARIANA GRÁVIDA SUMIU E ME LARGOU COM O BEBÊ, E AGORA...",
        "comentario_pai": "",
        "comentario": "Cadê a Maria??",
        "perguntas": [],
    },
    {
        "titulo": "Crescimento da barriga e dor no pé da barriga com 19 semanas de gravidez",
        "comentario_pai": "",
        "comentario": "Boa noite Patrícia eu estou grávida de 13 semanas mas não sinto nada de barriga endurecer a minha barriga está bem mole isso é normal",
        "perguntas": [
            "É normal que a barriga de uma gestante com 13 semanas de gravidez permaneça mole, sem endurecimento?"
        ],
    },
    {
        "titulo": "Teste do pezinho x teste da bochechinha",
        "comentario_pai": "",
        "comentario": "Tenho deficiência de G6PD, foi diagnosticado no exame do pezinho, ainda na maternidade quando nasci (1998), pelo SUS. Tive minha bebê há 3 meses, fiz o teste do pezinho pelo SUS também, mas não veio nada falando sobre G6PD. Há possibilidade dela ter a deficiência também? Pergunto pois, por conta da deficiência, não posso tomar a vacina de febre amarela e tenho receio da minha filha ter herdado. Tenho que fazer o teste da bochechinha? Outro ponto: cresci “sabendo” que a deficiência de G6PD não é uma doença, pois não tem “cura”, mas sim, uma deficiência.",
        "perguntas": [
            "Uma bebê de três meses cuja mãe tem deficiência de G6PD pode ter herdado essa deficiência?",
            "É necessário realizar o teste da bochechinha para verificar se uma bebê de três meses tem deficiência de G6PD?",
            "A deficiência de G6PD é considerada uma doença ou apenas uma deficiência genética?"
        ],
    },
]

In [6]:
EXAMPLES_FORMAT = []

for ex in EXTRACTION_EXAMPLES:
    user_content = json.dumps({
        "titulo": ex["titulo"],
        "comentario_pai": ex["comentario_pai"],
        "comentario": ex["comentario"]
    }, ensure_ascii=False)

    assistant_content = json.dumps(
        {"perguntas": ex["perguntas"]},
        ensure_ascii=False
    )

    EXAMPLES_FORMAT.append({"role": "user", "content": user_content})
    EXAMPLES_FORMAT.append({"role": "assistant", "content": assistant_content})

In [7]:
def extract_questions(
        comment_id: str,
        comentario: str,
        titulo: str,
        comentario_pai: str
    ) -> QuestionExtraction:

    user_content = json.dumps({
        "titulo": titulo,
        "comentario_pai": comentario_pai,
        "comentario": comentario
    }, ensure_ascii=False)

    messages = [
        *EXAMPLES_FORMAT,
        {"role": "user", "content": user_content},
    ]

    response = client.responses.parse(
        model=MODEL_NAME,
        instructions=EXTRACTION_PROMPT,
        input=messages,
        text_format=QuestionExtraction,
        reasoning={"effort": REASONING_EFFORT},
        service_tier=SERVICE_TIER
    )

    if LOG_USAGE:
        usage = response.usage
        if usage is None:
            raise ValueError(f"A API não retornou métricas de uso para {comment_id}")

        input_details = usage.input_tokens_details
        output_details = usage.output_tokens_details
        usage_row = pd.DataFrame([{
            "comment_id": comment_id,
            "model": MODEL_NAME,
            "input_tokens": usage.input_tokens,
            "cached_input_tokens": input_details.cached_tokens if input_details else 0,
            "output_tokens": usage.output_tokens,
            "reasoning_tokens": output_details.reasoning_tokens if output_details else 0,
            "total_tokens": usage.total_tokens
        }])
        usage_path = Path(USAGE_PATH)
        usage_path.parent.mkdir(parents=True, exist_ok=True)
        usage_row.to_csv(
            usage_path,
            mode='a',
            header=not usage_path.exists(),
            index=False
        )

    parsed_response = response.output_parsed
    if parsed_response is None:
        raise ValueError(f"Não foi possível interpretar a resposta para {comment_id}")

    return parsed_response

In [8]:
def extract_and_save(comment_id: str, comentario: str, titulo: str, comentario_pai: str, output_path: str=JSON_PATH):
    result = extract_questions(
        comment_id=comment_id,
        titulo=titulo,
        comentario_pai=comentario_pai,
        comentario=comentario
    )

    output = {
        'comment_id': comment_id,
        'comentario': comentario,
        'titulo': titulo,
        'comentario_pai': comentario_pai,
        'perguntas': result.perguntas
    }

    with jsonlines.open(output_path, 'a') as writer:
        writer.write(output)

In [9]:
processed = set()
try:
    with jsonlines.open(JSON_PATH, 'r') as reader:
        processed = {row["comment_id"] for row in reader}
except FileNotFoundError:
    pass

print(len(processed))

36319


In [10]:
for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing..."):
    comment_id = row['comment_id']
    comentario = "" if pd.isna(row['comentario']) else str(row['comentario'])
    titulo = "" if pd.isna(row['titulo']) else str(row['titulo'])
    comentario_pai = "" if pd.isna(row['comentario_pai']) else str(row['comentario_pai'])

    if comment_id in processed:
        continue

    extract_and_save(
        comment_id=comment_id,
        comentario=comentario,
        titulo=titulo,
        comentario_pai=comentario_pai
    )

Processing...:   0%|          | 0/35479 [00:00<?, ?it/s]

In [11]:
df_results = pd.read_json(JSON_PATH, lines=True)
df_results = df_results[df_results['comment_id'].isin(comments_ids)]
df_results.to_csv(OUTPUT_PATH, index=False)

# numero de comentários contendo perguntas relevantes
df_results[df_results['perguntas'].map(len) > 0].shape

(15997, 5)

In [12]:
df_sampled = df_results.sample(n=500, random_state=42)
df_sampled.to_csv(SAMPLED_PATH, index=False)